# Lab 2　神奇糖果（上）：把 regression 變成因果效果，然後看它歪掉

訓練師在進化前，會餵一些寶可夢吃「神奇糖果」（A = 1）。糖果到底讓進化後 CP 平均多多少？Lab 1 學會的 regression 怎麼變成這個答案（plug-in）？用了最會預測的 learner，答案為什麼還是歪的？對應投影片 37–53。

> **怎麼跑：** 這本筆記本用 **R** 執行（Colab 會自動選 R runtime；如果右上角不是 R，到「執行階段 → 變更執行階段類型」選 R）。從上往下依序執行：第一格下載資料、第二格載入工具箱，之後每一格都可以改了再跑。想一次跑完：「執行階段 → 全部執行」。
>
> 網頁版（不用登入、直接在瀏覽器跑）：[https://kiwi710680.github.io/Pokemon-CP/02_plugin.html](https://kiwi710680.github.io/Pokemon-CP/02_plugin.html)

In [ ]:
# Setup: download the data used by this lab (from https://raw.githubusercontent.com/kiwi710680/Pokemon-CP/main/) and set plot size
BASE <- "https://raw.githubusercontent.com/kiwi710680/Pokemon-CP/main/"
files <- c()
for (f in files) if (!file.exists(f)) {
  dir.create(dirname(f), recursive = TRUE, showWarnings = FALSE)
  for (try in 1:3) {                                   # retry: a single network hiccup should not stop the lab
    ok <- tryCatch(download.file(paste0(BASE, f), f, quiet = TRUE, mode = "wb") == 0, error = function(e) FALSE)
    if (ok) break else Sys.sleep(2)
  }
  if (!ok) stop("could not download ", f, " - check the internet connection and run this cell again")
}
options(repr.plot.width = 12, repr.plot.height = 7, repr.plot.res = 100, width = 110, digits = 4)
cat(if (length(files)) paste("ready:", length(files), "data file(s)") else "ready: this lab simulates its own data", "\n")

## 工具箱

這一頁用到的 R 函式都在下面這一格：估計器、資料讀取、畫圖小工具。先執行它；想知道某個函式做什麼，回來這裡找。

In [ ]:
# ---- estimators: one set of nuisance predictions -> different answers ----
# Notation (same as the slides): O = (X, A, Y)
#   Q(a, X) = E[Y | A = a, X]            outcome regression   -> nu$Q1, nu$Q0
#   g(X)    = P(A = 1 | X)               propensity score     -> nu$g
#   H(A, X) = A / g - (1 - A) / (1 - g)  clever covariate
#   D(O; g, Q, psi) = H (Y - Q_A) + Q1 - Q0 - psi   efficient influence function of the ATE
expit <- function(x) 1 / (1 + exp(-x))
bound <- function(p, lo = 0.025, hi = 0.975) pmin(pmax(p, lo), hi)

est_plugin <- function(nu) mean(nu$Q1 - nu$Q0)               # g-computation: predict twice, subtract, average

est_ipw_ht <- function(dat, nu)                               # Horvitz-Thompson IPTW: divide by n
  mean(dat$A * dat$Y / nu$g - (1 - dat$A) * dat$Y / (1 - nu$g))

est_ipw <- function(dat, nu) {                                # Hajek (stabilized) IPTW: divide by the sum of weights
  w1 <- dat$A / nu$g; w0 <- (1 - dat$A) / (1 - nu$g)
  sum(w1 * dat$Y) / sum(w1) - sum(w0 * dat$Y) / sum(w0)
}

eif <- function(dat, nu, psi) {
  H  <- dat$A / nu$g - (1 - dat$A) / (1 - nu$g)
  QA <- ifelse(dat$A == 1, nu$Q1, nu$Q0)
  H * (dat$Y - QA) + nu$Q1 - nu$Q0 - psi
}

wald <- function(psi, D) {                                    # SE from the EIF: sqrt(mean(D^2) / n)
  se <- sqrt(mean(D^2) / length(D))
  c(est = psi, se = se, lo = psi - 1.96 * se, hi = psi + 1.96 * se)
}

est_aipw <- function(dat, nu) {                               # one-step: psi_plug + P_n D  (solves P_n D = 0)
  psi <- mean(eif(dat, nu, 0))
  wald(psi, eif(dat, nu, psi))
}

est_tmle <- function(dat, nu, ybounds = range(dat$Y)) {       # targeting: move Q-hat along H until P_n D = 0
  lo <- ybounds[1]; hi <- ybounds[2]
  sc <- function(y) pmin(pmax((y - lo) / (hi - lo), 1e-6), 1 - 1e-6)
  Ys <- (dat$Y - lo) / (hi - lo)
  Q1s <- sc(nu$Q1); Q0s <- sc(nu$Q0); QAs <- ifelse(dat$A == 1, Q1s, Q0s)
  H  <- dat$A / nu$g - (1 - dat$A) / (1 - nu$g)
  fl <- suppressWarnings(glm(Ys ~ -1 + H, offset = qlogis(QAs), family = quasibinomial()))
  eps <- unname(coef(fl))
  Q1t <- expit(qlogis(Q1s) + eps / nu$g)
  Q0t <- expit(qlogis(Q0s) - eps / (1 - nu$g))
  nu_t <- data.frame(Q1 = lo + (hi - lo) * Q1t, Q0 = lo + (hi - lo) * Q0t, g = nu$g)
  psi <- mean(nu_t$Q1 - nu_t$Q0)
  c(wald(psi, eif(dat, nu_t, psi)), eps = eps)
}

weight_summary <- function(dat, nu) {                         # largest weight and effective sample size (ESS)
  w <- ifelse(dat$A == 1, 1 / nu$g, 1 / (1 - nu$g))
  c(max_weight = max(w),
    ESS_treated = sum(w[dat$A == 1])^2 / sum(w[dat$A == 1]^2), n_treated = sum(dat$A),
    ESS_control = sum(w[dat$A == 0])^2 / sum(w[dat$A == 0]^2), n_control = sum(1 - dat$A))
}

# ---- magic candy: simulated data, so the truth (Q, g, psi) is known ----
species_levels <- c("Pidgey", "Weedle", "Caterpie", "Eevee")

simulate_candy <- function(n = 800, seed = NULL, strength = 1) {
  if (!is.null(seed)) set.seed(seed)
  species <- factor(sample(species_levels, n, replace = TRUE, prob = c(.40, .20, .15, .25)), levels = species_levels)
  top <- c(Pidgey = 450, Weedle = 260, Caterpie = 260, Eevee = 650)[as.character(species)]
  cp  <- round(10 + top * rbeta(n, 2, 2.5))
  mult <- c(Pidgey = 1.85, Weedle = 1.10, Caterpie = 1.08, Eevee = 2.60)[as.character(species)]
  f0   <- mult * cp * (1 - cp / 2600)                         # evolution curve, bends at high CP
  # trainers prefer strong Pokemon and Eevee: confounding. Larger strength -> g closer to 0 or 1
  g0 <- expit(strength * (-1.8 + 3.2 * (cp / top)) + 0.5 * (species == "Eevee"))
  A  <- rbinom(n, 1, g0)
  tau <- 20 + 0.30 * cp                                       # candy effect: stronger Pokemon gain more
  Y1 <- f0 + tau + rnorm(n, 0, 30)
  Y0 <- f0 + rnorm(n, 0, 30)
  data.frame(species, cp, A, Y = ifelse(A == 1, Y1, Y0), g0, Q1 = f0 + tau, Q0 = f0, Y1, Y0)
}

candy_truth <- function(n = 2e6, seed = 1) {                  # population ATE = E[tau(X)]
  set.seed(seed)
  species <- sample(species_levels, n, replace = TRUE, prob = c(.40, .20, .15, .25))
  top <- c(Pidgey = 450, Weedle = 260, Caterpie = 260, Eevee = 650)[species]
  mean(20 + 0.30 * round(10 + top * rbeta(n, 2, 2.5)))
}
candy_population <- function(n = 20000, seed = 2027, strength = 1) simulate_candy(n, seed, strength)

# ---- nuisance learners ----
knn_Q <- function(train, newdata, a, k = 10) {                # same species, same arm, k nearest CP, average Y
  out <- rep(NA_real_, nrow(newdata))
  for (s in levels(newdata$species)) {
    tr <- train[train$species == s & train$A == a, ]
    id <- which(newdata$species == s)
    if (length(id) == 0) next
    if (nrow(tr) == 0) { out[id] <- mean(train$Y[train$A == a]); next }
    kk <- min(k, nrow(tr))
    D  <- abs(outer(newdata$cp[id], tr$cp, "-"))
    out[id] <- apply(D, 1, function(d) mean(tr$Y[order(d)[seq_len(kk)]]))
  }
  out
}
lin_Q <- function(train, newdata, a) predict(lm(Y ~ A + cp, data = train), transform(newdata, A = a))  # one straight line
logit_g <- function(train, newdata) {                        # each species its own intercept and slope
  fit <- glm(A ~ species + species:cp, family = binomial, data = train)
  bound(predict(fit, newdata, type = "response"))
}

fit_nuisance <- function(train, newdata, Qlearner = "knn", k = 10) {
  if (Qlearner == "knn") { Q1 <- knn_Q(train, newdata, 1, k); Q0 <- knn_Q(train, newdata, 0, k) }
  else { Q1 <- lin_Q(train, newdata, 1); Q0 <- lin_Q(train, newdata, 0) }
  data.frame(Q1 = Q1, Q0 = Q0, g = logit_g(train, newdata))
}
nuisance_full <- function(dat, ...) fit_nuisance(dat, dat, ...)          # train and predict on the same Pokemon
nuisance_crossfit <- function(dat, K = 5, seed = NULL, ...) {             # each prediction comes from the other folds
  if (!is.null(seed)) set.seed(seed)
  fold <- sample(rep(seq_len(K), length.out = nrow(dat)))
  nu <- data.frame(Q1 = NA_real_, Q0 = NA_real_, g = NA_real_)[rep(1, nrow(dat)), ]
  for (k in seq_len(K)) { te <- fold == k; nu[te, ] <- fit_nuisance(dat[!te, ], dat[te, ], ...) }
  rownames(nu) <- NULL
  nu
}

all_estimates <- function(dat, k = 10, K = 5) {
  nf <- nuisance_full(dat, k = k); nc <- nuisance_crossfit(dat, K = K, k = k)
  rbind(naive       = c(est = mean(dat$Y[dat$A == 1]) - mean(dat$Y[dat$A == 0]), se = NA, lo = NA, hi = NA),
        linear      = c(est = unname(coef(lm(Y ~ A + cp, data = dat))["A"]), se = NA, lo = NA, hi = NA),
        plugin_knn  = c(est = est_plugin(nf), se = NA, lo = NA, hi = NA),
        ipw         = c(est = est_ipw(dat, nf), se = NA, lo = NA, hi = NA),
        aipw_full   = est_aipw(dat, nf),
        tmle_full   = est_tmle(dat, nf)[1:4],
        aipw_cf_DML = est_aipw(dat, nc),
        tmle_cf     = est_tmle(dat, nc)[1:4])
}

# ---- only a simulation can do this: split the error into pieces ----
#   plug-in: psi_plug - psi = sampling + first-order bias + empirical process
#   AIPW   : psi_AIPW - psi = (P_n - P)D + R2 + empirical process
# "P" (population mean) is approximated by a large population `pop`
decompose_candy <- function(dat, pop, k = 10, scheme = c("full", "crossfit"), K = 5, seed = 1) {
  scheme <- match.arg(scheme)
  psi <- candy_truth(); n <- nrow(dat)
  set.seed(seed)
  fold <- if (scheme == "full") rep(1L, n) else sample(rep(seq_len(K), length.out = n))
  nu <- data.frame(Q1 = numeric(n), Q0 = numeric(n), g = numeric(n))
  first_order <- 0; R2 <- 0; gerr <- 0; qerr <- 0
  for (f in unique(fold)) {
    te <- fold == f
    tr <- if (scheme == "full") rep(TRUE, n) else !te
    nu[te, ] <- fit_nuisance(dat[tr, ], dat[te, ], k = k)
    np <- fit_nuisance(dat[tr, ], pop, k = k)                 # the same model, evaluated on the population
    w <- mean(te)
    r2_unit <- (np$g - pop$g0) / np$g * (np$Q1 - pop$Q1) + (np$g - pop$g0) / (1 - np$g) * (np$Q0 - pop$Q0)
    fo_unit <- (np$Q1 - pop$Q1) - (np$Q0 - pop$Q0)
    first_order <- first_order + w * mean(fo_unit)
    R2 <- R2 + w * mean(r2_unit)
    if (f == fold[1]) units <- data.frame(species = pop$species, cp = pop$cp, g_err = np$g - pop$g0,
                                          Q1_err = np$Q1 - pop$Q1, Q0_err = np$Q0 - pop$Q0,
                                          R2_unit = r2_unit, first_order_unit = fo_unit)
    gerr <- gerr + w * mean((np$g - pop$g0)^2)
    qerr <- qerr + w * mean(((np$Q1 - pop$Q1)^2 + (np$Q0 - pop$Q0)^2) / 2)
  }
  plug <- mean(nu$Q1 - nu$Q0)
  aipw <- est_aipw(dat, nu)[["est"]]
  samp_plug <- mean(dat$Q1 - dat$Q0) - psi
  samp_aipw <- mean(eif(dat, data.frame(Q1 = dat$Q1, Q0 = dat$Q0, g = dat$g0), psi))   # (P_n - P)D, since PD = 0
  list(plugin = c(total = plug - psi, sampling = samp_plug, first_order = first_order,
                  emp_process = plug - psi - samp_plug - first_order),
       aipw = c(total = aipw - psi, sampling = samp_aipw, R2 = R2, emp_process = aipw - psi - samp_aipw - R2),
       L2 = c(g = sqrt(gerr), Q = sqrt(qerr), product = sqrt(gerr) * sqrt(qerr)),
       units = units)                                         # per-Pokemon errors (first fold's model), for plots
}

# Neyman orthogonality: move g and Q away from the truth by t, compare the two moments
orthogonality_curves <- function(pop, ts = seq(-1, 1, by = 0.05), dq1 = 40, dq0 = -20, dg = 0.6) {
  g <- pop$g0; Q1 <- pop$Q1; Q0 <- pop$Q0
  psi <- mean(Q1 - Q0)
  gbar <- g + dg * g * (1 - g)
  eif_moment <- function(t, move_g, move_Q) {
    gt <- if (move_g) g + t * (gbar - g) else g
    q1 <- if (move_Q) Q1 + t * dq1 else Q1
    q0 <- if (move_Q) Q0 + t * dq0 else Q0
    mean(q1 - q0 + g / gt * (Q1 - q1) - (1 - g) / (1 - gt) * (Q0 - q0)) - psi
  }
  data.frame(t = ts, plugin_Q = ts * (dq1 - dq0),
             eif_Q_only = sapply(ts, eif_moment, move_g = FALSE, move_Q = TRUE),
             eif_g_only = sapply(ts, eif_moment, move_g = TRUE, move_Q = FALSE),
             eif_both   = sapply(ts, eif_moment, move_g = TRUE, move_Q = TRUE))
}

# smallest possible SE: sqrt(V_eff / n), V_eff = E[sigma^2/g + sigma^2/(1-g)] + Var(tau), sigma = 30
candy_se_bound <- function(pop, n) sqrt((mean(30^2 / pop$g0 + 30^2 / (1 - pop$g0)) + var(pop$Q1 - pop$Q0)) / n)

# ---- plotting helpers (base graphics only) ----
COL <- list(red = "#D9480F", teal = "#0B7285", ink = "#1C1B19", muted = "#6B665E",
            gold = "#C98A0C", plum = "#7B3F8C", rule = "#DDD6CA")
lab_par <- function(...) par(mar = c(4.5, 5, 1.5, 1), las = 1, bty = "n", col.axis = COL$muted, ...)

overlap_plot <- function(g_raw, A, xlab = "estimated propensity score  g-hat(X)", labels = c("A = 1", "A = 0"), breaks = 40) {
  br <- seq(0, 1, length.out = breaks + 1)                    # mirrored histogram: treated up, control down
  h1 <- hist(pmin(pmax(g_raw[A == 1], 0), 1), breaks = br, plot = FALSE)$counts
  h0 <- hist(pmin(pmax(g_raw[A == 0], 0), 1), breaks = br, plot = FALSE)$counts
  lab_par()
  plot(NA, xlim = c(0, 1), ylim = c(-max(h0), max(h1)), axes = FALSE, xlab = xlab, ylab = "count")
  axis(1); yt <- pretty(c(-max(h0), max(h1))); axis(2, at = yt, labels = abs(yt))
  rect(br[-length(br)], 0, br[-1], h1, col = adjustcolor(COL$red, .8), border = "white")
  rect(br[-length(br)], 0, br[-1], -h0, col = adjustcolor(COL$teal, .8), border = "white")
  abline(h = 0)
  text(0.98, max(h1) * .85, labels[1], adj = 1, col = COL$red, font = 2)
  text(0.98, -max(h0) * .85, labels[2], adj = 1, col = COL$teal, font = 2)
}

forest_plot <- function(lab, est, lo = NA, hi = NA, ref = NULL, ref_lab = "", col = COL$ink, xlab = "estimate", xlim = NULL) {
  n <- length(est); y <- n:1
  rng <- range(c(est, lo, hi, ref), na.rm = TRUE)
  if (is.null(xlim)) xlim <- rng + c(-1, 1) * diff(rng) * 0.08
  op <- par(mar = c(4.5, min(22, max(8, 0.62 * max(nchar(lab)) + 1)), 1.5, 1), las = 1, bty = "n", col.axis = COL$muted)
  on.exit(par(op))
  plot(NA, xlim = xlim, ylim = c(0.5, n + 0.5), axes = FALSE, xlab = xlab, ylab = "")
  abline(h = y, col = "#F0EBE3")
  if (!is.null(ref)) {
    abline(v = ref, lty = 2, col = COL$muted); pos <- (ref[1] - xlim[1]) / diff(xlim)
    mtext(ref_lab, 3, at = ref[1], adj = if (pos > 0.7) 1 else if (pos < 0.3) 0 else 0.5, cex = .8, col = COL$muted)
  }
  ok <- !is.na(lo)
  segments(lo[ok], y[ok], hi[ok], y[ok], lwd = 3, col = rep_len(col, n)[ok])
  points(est, y, pch = 19, cex = 1.5, col = rep_len(col, n))
  axis(1); axis(2, at = y, labels = lab, tick = FALSE, col.axis = COL$ink)
}

| 格 | 投影片 | 這格回答的問題 |
|---|---|---|
| 01–02 | 37–44 | 誰吃到糖？直接相減錯在哪？ |
| 03–04 | 45–47 | 因果問題怎麼變成 regression？plug-in 是什麼？Q 該用哪個模型？ |
| 05–06 | 48–51 | 為什麼預測很準，ATE 還是歪？歪的那一塊有多大？ |
| 07 | 52–53 | 另一條路 IPTW：只信 g 會怎樣？ |

> **虛構情境、模擬資料。**因為資料是我們自己生成的，**真值 ψ ≈ 80.8**，真的 Q 和 g 也都知道——所以可以做真實研究做不到的事：直接看答案錯了多少、錯在哪裡。Lab 5–8 換成真實資料，就沒有這個奢侈了。

## 01　誰吃到糖？　<small>（投影片 43–44）</small>

X = (物種, 進化前 CP)，A = 有沒有吃糖，Y = 進化後 CP。訓練師偏心：CP 高的、伊布，比較容易吃到糖。CP 本身又決定進化後 CP。X 同時影響 A 和 Y——這就是 confounding。

In [ ]:
truth <- candy_truth()                     # population truth psi = E[Y(1) - Y(0)]
d <- simulate_candy(1500, seed = 2026)
head(d[, c("species", "cp", "A", "Y")])
table(species = d$species, Candy = d$A)
round(tapply(d$cp, d$A, mean), 1)           # Pokemon that got candy start with higher CP
pg <- d[d$species == "Pidgey", ]
lab_par()
plot(pg$cp, pg$Y, pch = 19, col = ifelse(pg$A == 1, COL$red, COL$teal), xlab = "Original CP (Pidgey)", ylab = "CP after evolution")
legend("topleft", c("Candy A = 1", "No Candy A = 0"), pch = 19, col = c(COL$red, COL$teal), bty = "n")

> **看什麼：**
>
> 紅點（吃糖）集中在右上——吃糖的本來就比較強。直接比較兩組，會把「本來就比較強」算成糖果的功勞。

## 02　直覺做法：直接相減、一條直線　<small>（投影片 44、47）</small>

In [ ]:
c(truth = truth,
  naive_difference = mean(d$Y[d$A == 1]) - mean(d$Y[d$A == 0]),
  linear_adjusted = unname(coef(lm(Y ~ A + cp, data = d))["A"]))

> **看什麼：**
>
> 直接相減 229，差了快三倍（confounding）。加一條直線調整 CP 得到 63，又往另一邊偏（model misspecification：物種不同、關係會彎、糖果效果還跟著 CP 變大，直線放不下）。

## 03　Plug-in：同一隻寶可夢，預測兩次　<small>（投影片 45–46）</small>

識別假設成立時，`ψ = E[Q₁(X) − Q₀(X)]`，Q 就是 Lab 1 的 regression。先學 Q̂（這裡用 kNN：同物種、同一組裡找 CP 最接近的 10 隻取平均），對每一隻「假裝吃糖」和「假裝沒吃」各預測一次，相減、平均，就結束了。這叫 g-computation，也叫 plug-in：`ψ̂ = (1/n) Σ {Q̂₁(Xᵢ) − Q̂₀(Xᵢ)}`。順便也把另一個 nuisance——propensity score `g(X) = P(A = 1 | X)`——估起來，第 07 格會用到。

In [ ]:
nf <- nuisance_full(d, k = 10)               # Q1 = Q-hat(1, X), Q0 = Q-hat(0, X) via kNN; g = g-hat(X) via logistic regression
head(cbind(d[, c("species", "cp", "A", "Y")], Q1 = round(nf$Q1, 1), Q0 = round(nf$Q0, 1), diff = round(nf$Q1 - nf$Q0, 1)))
c(truth = truth, plugin_kNN = est_plugin(nf))

> **看什麼：**
>
> 每一列都有 Q1、Q0：吃糖的那隻，Q0 是「如果牠沒吃」的預測；沒吃的那隻，Q1 是「如果牠吃了」。diff 欄平均起來就是答案 84.2——比一條直線好很多，但還是比真值多了 3.4。

## 04　那 Q 要用哪一個模型？kNN 在哪裡猜錯　<small>（投影片 47）</small>

低 CP 的波波很少吃糖。要猜「低 CP 波波**如果**吃了糖」，kNN 找不到夠近的鄰居，只好拿比較強的來充數。

In [ ]:
xs <- data.frame(species = factor("Pidgey", levels = species_levels), cp = 10:460, A = 1, Y = 0)
q_knn  <- knn_Q(d, xs, a = 1, k = 10)
q_true <- 1.85 * xs$cp * (1 - xs$cp / 2600) + 20 + 0.30 * xs$cp
lab_par()
plot(pg$cp[pg$A == 1], pg$Y[pg$A == 1], pch = 19, col = adjustcolor(COL$red, .5), xlim = c(0, 470),
     xlab = "Original CP (Pidgey)", ylab = "CP after evolution if eat Candy")
lines(xs$cp, q_true, lwd = 3, lty = 2); lines(xs$cp, q_knn, lwd = 3, col = COL$red, type = "s")
rug(pg$cp[pg$A == 1], col = COL$red)
legend("topleft", c("true Q1(x)", "kNN Q1-hat(x)"), lty = c(2, 1), lwd = 3, col = c(COL$ink, COL$red), bty = "n")

> **看什麼：**
>
> 下方短線是吃糖的波波，左邊很稀疏。那裡紅色階梯（Q̂₁）明顯高於虛線（Q₁）。這種寶可夢不多，預測 loss 幾乎感覺不到；但 plug-in 要替**每一隻**預測另一個世界，偏差就一路傳到 ψ̂。

## 05　千萬不要說：「預測 loss 很低，plug-in 就準」　<small>（投影片 48）</small>

Lab 1 的做法是：挑預測 loss 最小的模型。那就把它用在這裡——另外抽 2,000 隻寶可夢當測試資料，比較一條直線和不同 k 的 kNN：誰預測最準？誰的 ATE 最準？

In [ ]:
test <- simulate_candy(2000, seed = 99)                       # fresh Pokemon, never used for training
learners <- list(linear = 0, kNN_k1 = 1, kNN_k3 = 3, kNN_k10 = 10, kNN_k30 = 30, kNN_k100 = 100)
cmp <- t(sapply(learners, function(k) {
  Qa <- function(nd, a) if (k == 0) lin_Q(d, nd, a) else knn_Q(d, nd, a, k)
  pred <- ifelse(test$A == 1, Qa(test, 1), Qa(test, 0))       # predict the outcome we can see
  plug <- mean(Qa(d, 1) - Qa(d, 0))                           # plug-in ATE on the study sample
  c(test_RMSE = sqrt(mean((test$Y - pred)^2)), plugin_ATE = plug, ATE_error = plug - truth)
}))
round(cmp, 2)
best <- which.min(cmp[, "test_RMSE"])                         # what Lab 1's rule would pick
lab_par(mar = c(5.2, 6, 1.5, 1), mgp = c(4, .8, 0))
plot(cmp[, "test_RMSE"], abs(cmp[, "ATE_error"]), pch = 19, cex = 2, col = COL$red, log = "y", xlim = c(30, 125),
     xlab = "prediction error on new Pokemon (RMSE)", ylab = "|ATE error| (log scale)")
points(cmp[best, "test_RMSE"], abs(cmp[best, "ATE_error"]), cex = 4, lwd = 2)
text(cmp[, "test_RMSE"], abs(cmp[, "ATE_error"]), sub("_", " ", rownames(cmp)), pos = 4, offset = 1, col = COL$ink)
legend("topleft", "lowest prediction error", pch = 1, pt.cex = 2.5, pt.lwd = 2, bty = "n")

> **看什麼：**
>
> - 預測最準的是 kNN k = 10（RMSE 約 33，圖上畫圈的那一點）——Lab 1 的規則會選它。可是它的 ATE 偏了 3.4。
> - k = 1、k = 3 預測比較差（RMSE 42、36），ATE 反而只偏 0.1–0.2。
> - 圖上的點沒有排成一條線：**預測 loss 的排名，不是 ATE 誤差的排名。**

<details><summary><b>想一想：為什麼會這樣？投影片 48 的三個原因，各對應這張圖的哪個地方？</b></summary>

原因 1（考的範圍不一樣）：RMSE 平均的是「看得到的那一臂」，ATE 要預測「另一個世界」，而且偏偏落在吃糖很少的低 CP 那一區。原因 2（ML 故意偏一點）：k 越大越平滑，預測的 variance 小了，但平滑造成的偏差會原封不動傳進 ψ̂——k = 30、100 就是這樣。原因 3（自己出題自己改）：這一格的 Q̂ 是用同一份資料訓練又預測；k = 1 時最近的鄰居常常就是自己，Lab 4 第 05 格會看到它怎麼騙過 SE。所以「挑 loss 最小的 learner」是對預測的最佳化，不是對 ψ 的最佳化。

</details>

## 06　把 plug-in 的誤差拆成三塊　<small>（投影片 49–51）</small>

那 k = 10 偏掉的 3.4 是從哪裡來的？投影片 49 的式子：

`ψ̂_plug − ψ = (ℙn − P){Q₁ − Q₀}　+　P{(Q̂₁ − Q₁) − (Q̂₀ − Q₀)}　+　(ℙn − P){(Q̂₁ − Q₁) − (Q̂₀ − Q₀)}`

真實研究裡這三塊都看不到。但模擬資料知道真的 Q，「母體平均 P」可以用另外 8,000 隻同分布的寶可夢近似——所以三塊都算得出來（約 20 秒）。

In [ ]:
pop <- candy_population(8000)
dec <- decompose_candy(d, pop, k = 10, scheme = "full")
round(dec$plugin, 2)
lab_par(mar = c(4.5, 11, 1, 1))
barplot(rev(dec$plugin), horiz = TRUE, names.arg = rev(c("total error", "sampling error", "first-order bias", "empirical process")),
        col = rev(c(COL$ink, COL$teal, COL$red, COL$plum)), border = NA, xlab = "error of the estimate (CP)")
abline(v = 0)

> **看什麼：**
>
> - 總誤差約 3.4，最大的一塊是紅色的 first-order bias（約 4.8）——Q̂ 估錯、平均之後跟著偏的那一塊，它的大小跟 ‖Q̂ − Q‖ 同一個量級。
> - 藍色的抽樣誤差（約 0.2）本來就該有，是 CI 要描述的東西。
> - 紫色（約 −1.6）是「同一份資料又訓練又評分」留下來的那一塊。
> - 所以「把預測做準」不夠：ML 學得多慢，first-order bias 就多大。怎麼辦？Lab 3 會把這一塊估出來、扣掉。

## 07　另一條路：IPTW　<small>（投影片 52–53）</small>

如果不信 Q̂，只信 ĝ 呢？很少吃糖的那類，少數吃了糖的要代表更多同類：權重 1/ĝ。HT 版本直接乘權重除以 n；Hájek（stabilized）版本除以各組權重總和。

In [ ]:
c(truth = truth, IPTW_HT = est_ipw_ht(d, nf), IPTW_Hajek = est_ipw(d, nf))
round(weight_summary(d, nf), 1)        # largest weight and effective sample size (ESS) per arm
overlap_plot(nf$g, d$A, labels = c("candy", "no candy"))

> **看什麼：**
>
> ĝ 從 0.09 到 0.87，兩組廣泛重疊；最大權重約 10，有效樣本數掉了一些但還健康。這次 IPTW（約 77.5）往另一邊偏了 3。plug-in 只信 Q̂，IPTW 只信 ĝ——兩個都是「只買一張保險」。

## 08　帶走一句話

### 預測最準的 learner，不等於 ATE 最準；plug-in 的偏差跟著 Q̂ 的誤差走。

兩張保險各自都會歪。那能不能兩張一起買，而且讓 Q̂ 的錯誤不要一路傳進答案？Lab 3：用 EIF 把一階誤差估出來、扣掉。